# 🔍 Data Science Is a Recipe — Live Fraud Demo
### `Outcome = f(Problem, Data, Model, Evaluation, Explainability)`

This notebook walks the full DS recipe using synthetic fraud data — every cell is a chapter of the detective story.

**Install once if needed:**
```bash
pip install faker catboost xgboost shap imbalanced-learn scikit-learn pandas numpy matplotlib seaborn
```

In [ ]:
# ── IMPORTS ────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from faker import Faker
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve,
    roc_curve, confusion_matrix, classification_report, brier_score_loss
)
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

# Palette matching the deck
C = dict(
    navy='#1E2D4E', teal='#028090', mint='#02C39A',
    purple='#5B2C8D', orange='#BF5E00', red='#C0392B',
    light='#EFF4F8', muted='#8899AA'
)

plt.rcParams.update({
    'figure.facecolor': '#141D30',
    'axes.facecolor':   '#1E2D4E',
    'axes.edgecolor':   '#3A4F70',
    'axes.labelcolor':  '#CCDDEE',
    'xtick.color':      '#8899AA',
    'ytick.color':      '#8899AA',
    'text.color':       '#CCDDEE',
    'grid.color':       '#2A3D5A',
    'grid.alpha':       0.5,
    'font.family':      'sans-serif',
})

SEED = 42
np.random.seed(SEED)
print('✅  Libraries loaded')

---
## Step 1 — The Problem
> *"A transaction just came through. Approve or decline? You have 200ms."*

We're building a **fraud classifier** — binary classification. The recipe is `Classify`.
Primary metric: **PR-AUC** (not accuracy — class imbalance makes accuracy useless here).

---
## Step 2 — The Data (with Faker)
> *"What slice of reality did you show the algorithm?"*

We generate realistic transaction records using `Faker` for the human-readable layer
and `make_classification` for the latent fraud signal.

In [ ]:
# ── GENERATE SYNTHETIC FRAUD DATA ─────────────────────────────────────────────
fake = Faker()
Faker.seed(SEED)
N = 50_000
FRAUD_RATE = 0.012   # 1.2% — realistic for card fraud

# Latent features via make_classification (gives us correlated signal)
X_latent, y = make_classification(
    n_samples=N,
    n_features=8,
    n_informative=5,
    n_redundant=2,
    n_clusters_per_class=2,
    weights=[1 - FRAUD_RATE, FRAUD_RATE],
    flip_y=0.02,          # 2% label noise — real fraud data is messy
    class_sep=0.85,
    random_state=SEED
)

# Scale latent features to plausible real-world ranges
rng = np.random.default_rng(SEED)

txn_amount       = np.abs(X_latent[:, 0] * 800 + 200).clip(1, 15000)
geo_distance_mi  = np.abs(X_latent[:, 1] * 400 + 50).clip(0, 3000)
txn_velocity_1h  = np.abs(X_latent[:, 2] * 4 + 2).clip(1, 30).astype(int)
amount_vs_avg    = np.abs(X_latent[:, 3] * 3 + 1).clip(0.05, 20)
account_age_days = np.abs(X_latent[:, 4] * 600 + 800).clip(1, 3650).astype(int)
# Fraud tends to happen late at night — add 0-7 hour offset for fraud rows
hour_of_day      = (rng.integers(0, 24, N) + y * rng.integers(0, 8, N)) % 24
merchant_risk    = np.abs(X_latent[:, 5] * 0.4 + 0.3).clip(0, 1)
intl_flag        = (geo_distance_mi > 500).astype(int)

merchant_cats = ['Groceries','Gas','Electronics','Restaurant','Travel',
                 'ATM','Online','Jewelry','Gaming','Healthcare']
merchant_cat  = rng.choice(merchant_cats, N)

df = pd.DataFrame({
    'txn_id':          [fake.uuid4()[:8].upper() for _ in range(N)],
    'customer_id':     [f'CUST-{rng.integers(1000,9999)}' for _ in range(N)],
    'txn_amount':      txn_amount.round(2),
    'geo_distance_mi': geo_distance_mi.round(1),
    'txn_velocity_1h': txn_velocity_1h,
    'amount_vs_avg':   amount_vs_avg.round(3),
    'account_age_days':account_age_days,
    'hour_of_day':     hour_of_day,
    'merchant_risk':   merchant_risk.round(4),
    'intl_flag':       intl_flag,
    'merchant_cat':    merchant_cat,
    'is_fraud':        y
})

fraud_n  = df['is_fraud'].sum()
legit_n  = N - fraud_n
print(f'Dataset: {N:,} transactions')
print(f'  Fraud:  {fraud_n:,}  ({fraud_n/N*100:.2f}%)')
print(f'  Legit:  {legit_n:,}  ({legit_n/N*100:.2f}%)')
df.head(6)

---
## Step 3 — Sampling: What You Choose to Learn From
> *"Bad sample → bad model. Before models, before metrics — what slice of reality did you show the algorithm?"*

**Three sampling traps:**
1. **Class imbalance** — 1.2% fraud means naïve accuracy = 98.8% (by predicting NOTHING)
2. **Survivorship bias** — we only see *attempted* fraud. Blocked fraud is invisible.
3. **Temporal leakage** — random splits let future signals leak into training. Always split by time.

In [ ]:
# ── VISUALIZE CLASS IMBALANCE ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Step 3 — Sampling: What You Choose to Learn From', 
             fontsize=13, fontweight='bold', color='white', y=1.02)

# Panel 1: Class distribution
ax = axes[0]
counts = df['is_fraud'].value_counts()
bars = ax.bar(['Legitimate', 'Fraud'], counts.values,
              color=[C['teal'], C['red']], width=0.5, edgecolor='none')
ax.set_title('Raw Class Distribution', color='white', fontsize=10, pad=8)
ax.set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontsize=9, color='white')
ax.text(0.98, 0.95, f'Fraud rate:\n{fraud_n/N*100:.2f}%', transform=ax.transAxes,
        ha='right', va='top', fontsize=9, color=C['orange'],
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#2A0A00', edgecolor=C['orange']))

# Panel 2: Naive accuracy trap
ax = axes[1]
naive_acc = legit_n / N * 100
ax.barh(['Predict Everything\nLegit (baseline)', 'Our Target\n(PR-AUC focus)'],
        [naive_acc, 72], color=[C['muted'], C['mint']], height=0.4)
ax.axvline(50, color=C['orange'], linestyle='--', linewidth=1.2, alpha=0.7)
ax.set_xlim(0, 105)
ax.set_xlabel('Accuracy %')
ax.set_title('The Accuracy Trap', color='white', fontsize=10, pad=8)
ax.text(naive_acc - 1, 0, f'{naive_acc:.1f}%', va='center', ha='right',
        color='white', fontsize=9, fontweight='bold')
ax.text(73, 1, '~72% PR-AUC', va='center', color=C['mint'], fontsize=9)
ax.text(52, -0.35, '50% random', color=C['orange'], fontsize=8)

# Panel 3: Feature distributions by class
ax = axes[2]
fraud_vel  = df.loc[df['is_fraud']==1, 'txn_velocity_1h']
legit_vel  = df.loc[df['is_fraud']==0, 'txn_velocity_1h']
ax.hist(legit_vel.clip(0,15), bins=15, density=True, alpha=0.6,
        color=C['teal'], label='Legit')
ax.hist(fraud_vel.clip(0,15), bins=15, density=True, alpha=0.75,
        color=C['red'], label='Fraud')
ax.set_title('txn_velocity_1h by Class', color='white', fontsize=10, pad=8)
ax.set_xlabel('Transactions in last hour')
ax.set_ylabel('Density')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'\n⚠️  Naive accuracy (predict all legit): {naive_acc:.2f}%')
print(f'   Fraud cases caught: 0 of {fraud_n:,}  — useless!')
print(f'\n✅  Use PR-AUC — it measures performance on the minority class')

In [ ]:
# ── FEATURE ENGINEERING + TRAIN/TEST SPLIT ────────────────────────────────────
# Always split temporally in production — we simulate with index as a time proxy
df_sorted = df.copy()

FEATURES = [
    'txn_amount', 'geo_distance_mi', 'txn_velocity_1h',
    'amount_vs_avg', 'account_age_days', 'hour_of_day',
    'merchant_risk', 'intl_flag'
]
TARGET = 'is_fraud'

split_idx = int(N * 0.75)
train_df  = df_sorted.iloc[:split_idx]
test_df   = df_sorted.iloc[split_idx:]

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]

print(f'Train: {len(X_train):,} rows  |  fraud: {y_train.sum():,}  ({y_train.mean()*100:.2f}%)')
print(f'Test:  {len(X_test):,}  rows  |  fraud: {y_test.sum():,}  ({y_test.mean()*100:.2f}%)')

# SMOTE on training set only (never touch test!)
smote = SMOTE(sampling_strategy=0.15, random_state=SEED)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f'\nAfter SMOTE:')
print(f'  Train: {len(X_train_bal):,} rows  |  fraud: {y_train_bal.sum():,}  ({y_train_bal.mean()*100:.2f}%)')
print(f'  (Test untouched — evaluate on real distribution)')

---
## Step 4 — Train the Model
> *"The algorithm is step 4 of 7. Getting steps 1–3 and 5–7 right matters more."*

We train three models to compare:
- **Logistic Regression** — interpretable baseline
- **XGBoost** — gradient boosted trees
- **XGBoost + SMOTE** — same model, better sampling

Then we'll also train a model *without* SMOTE to show the sampling effect.

In [ ]:
# ── TRAIN MODELS ──────────────────────────────────────────────────────────────
xgb_params = dict(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    scale_pos_weight=legit_n/fraud_n,   # class weight approach
    eval_metric='aucpr',
    random_state=SEED,
    verbosity=0
)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# Model 1: Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=500, random_state=SEED)
lr.fit(X_tr_sc, y_train)
lr_proba = lr.predict_proba(X_te_sc)[:, 1]

# Model 2: XGBoost (scale_pos_weight)
xgb_base = xgb.XGBClassifier(**xgb_params)
xgb_base.fit(X_train, y_train)
xgb_proba = xgb_base.predict_proba(X_test)[:, 1]

# Model 3: XGBoost + SMOTE (balanced sampling)
xgb_smote = xgb.XGBClassifier(
    **{**xgb_params, 'scale_pos_weight': 1}  # class weights baked into SMOTE
)
xgb_smote.fit(X_train_bal, y_train_bal)
xgb_smote_proba = xgb_smote.predict_proba(X_test)[:, 1]

models = {
    'Logistic Regression':  lr_proba,
    'XGBoost (weighted)':   xgb_proba,
    'XGBoost + SMOTE':      xgb_smote_proba,
}

print('Model            |  ROC-AUC  |  PR-AUC')
print('-' * 45)
for name, proba in models.items():
    roc = roc_auc_score(y_test, proba)
    pr  = average_precision_score(y_test, proba)
    print(f'{name:<24} |  {roc:.4f}  |  {pr:.4f}')

---
## Step 5 — Evaluation Is Multi-Dimensional
> *"Evaluation is not a number — it's a system of tradeoffs."*

Five dimensions we always check:
1. **Metrics (plural)** — PR-AUC, ROC-AUC, KS statistic
2. **Business tradeoffs** — false positive cost vs false negative cost at each threshold
3. **Latency** — predict on 10k rows, measure throughput
4. **Calibration** — does a score of 0.8 actually mean 80% fraud probability?
5. **Stability** — score distribution by decile

In [ ]:
# ── MULTI-DIMENSIONAL EVALUATION ──────────────────────────────────────────────
MAIN_MODEL  = 'XGBoost + SMOTE'
main_proba  = models[MAIN_MODEL]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(f'Step 5 — Multi-Dimensional Evaluation: {MAIN_MODEL}',
             fontsize=13, fontweight='bold', color='white', y=1.01)

COLORS = [C['teal'], C['mint'], C['orange']]

# ── 1. PR Curves ──
ax = axes[0, 0]
for (name, proba), color in zip(models.items(), COLORS):
    p, r, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(r, p, color=color, linewidth=2, label=f'{name} (PR-AUC={ap:.3f})')
baseline = y_test.mean()
ax.axhline(baseline, color=C['muted'], linestyle='--', linewidth=1,
           label=f'Random baseline ({baseline:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR Curves', color='white'); ax.legend(fontsize=7.5)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# ── 2. ROC Curves ──
ax = axes[0, 1]
for (name, proba), color in zip(models.items(), COLORS):
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1], color=C['muted'], linestyle='--', linewidth=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves', color='white'); ax.legend(fontsize=7.5)

# ── 3. Threshold: Precision vs Recall vs F1 ──
ax = axes[0, 2]
p_arr, r_arr, thresh = precision_recall_curve(y_test, main_proba)
f1_arr = np.where((p_arr + r_arr) > 0, 2*p_arr*r_arr/(p_arr+r_arr), 0)
ax.plot(thresh, p_arr[:-1], color=C['teal'],   linewidth=2, label='Precision')
ax.plot(thresh, r_arr[:-1], color=C['orange'], linewidth=2, label='Recall')
ax.plot(thresh, f1_arr[:-1],color=C['mint'],   linewidth=2, label='F1')
best_f1_idx = np.argmax(f1_arr[:-1])
best_thresh = thresh[best_f1_idx]
ax.axvline(best_thresh, color='white', linestyle=':', linewidth=1.2, alpha=0.7)
ax.text(best_thresh+0.01, 0.05, f'Best F1\nθ={best_thresh:.2f}',
        color='white', fontsize=8)
ax.set_xlabel('Decision Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold Analysis', color='white'); ax.legend(fontsize=8)

# ── 4. Business Cost at Each Threshold ──
ax = axes[1, 0]
FP_COST = 15    # $15 to review a false positive (analyst time)
FN_COST = 250   # $250 average fraud loss on false negative
thresholds = np.linspace(0.01, 0.99, 100)
total_costs = []
for t in thresholds:
    preds = (main_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    total_costs.append(fp * FP_COST + fn * FN_COST)
best_t_idx = np.argmin(total_costs)
ax.plot(thresholds, total_costs, color=C['purple'], linewidth=2)
ax.axvline(thresholds[best_t_idx], color=C['mint'], linestyle='--', linewidth=1.5,
           label=f'Min cost θ={thresholds[best_t_idx]:.2f}')
ax.set_xlabel('Decision Threshold'); ax.set_ylabel('Total Cost ($)')
ax.set_title(f'Business Cost (FP=${FP_COST} | FN=${FN_COST})', color='white')
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── 5. Score Distribution by Class ──
ax = axes[1, 1]
ax.hist(main_proba[y_test == 0], bins=60, density=True, alpha=0.6,
        color=C['teal'], label='Legit')
ax.hist(main_proba[y_test == 1], bins=60, density=True, alpha=0.75,
        color=C['red'], label='Fraud')
ax.axvline(best_thresh, color='white', linestyle=':', linewidth=1.5)
ax.set_xlabel('Predicted Fraud Probability')
ax.set_ylabel('Density')
ax.set_title('Score Separation', color='white')
ax.legend(fontsize=9)

# ── 6. KS Statistic ──
ax = axes[1, 2]
sorted_idx = np.argsort(main_proba)[::-1]
sorted_y   = y_test.values[sorted_idx]
total_fraud = sorted_y.sum()
total_legit = (1 - sorted_y).sum()
cum_fraud   = np.cumsum(sorted_y) / total_fraud
cum_legit   = np.cumsum(1 - sorted_y) / total_legit
pct_pop     = np.arange(1, len(sorted_y)+1) / len(sorted_y)
ks_vals     = cum_fraud - cum_legit
ks_stat     = ks_vals.max()
ks_idx      = ks_vals.argmax()
ax.plot(pct_pop*100, cum_fraud*100, color=C['red'],  linewidth=2, label='Fraud')
ax.plot(pct_pop*100, cum_legit*100, color=C['teal'], linewidth=2, label='Legit')
ax.vlines(pct_pop[ks_idx]*100, cum_legit[ks_idx]*100, cum_fraud[ks_idx]*100,
          colors=C['mint'], linewidth=2.5, label=f'KS={ks_stat:.3f}')
ax.set_xlabel('% Population (ranked by score)')
ax.set_ylabel('% Cumulative')
ax.set_title('KS Statistic', color='white'); ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Summary table
opt_t = thresholds[best_t_idx]
preds_opt = (main_proba >= opt_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, preds_opt).ravel()
print(f'\n📊 Evaluation Summary — {MAIN_MODEL}')
print(f'   PR-AUC:          {average_precision_score(y_test, main_proba):.4f}')
print(f'   ROC-AUC:         {roc_auc_score(y_test, main_proba):.4f}')
print(f'   KS Statistic:    {ks_stat:.4f}')
print(f'   Optimal threshold (cost): {opt_t:.3f}')
print(f'   At θ={opt_t:.2f}:  TP={tp:,}  FP={fp:,}  TN={tn:,}  FN={fn:,}')
print(f'   Fraud recall:    {tp/(tp+fn)*100:.1f}%  |  Precision: {tp/(tp+fp)*100:.1f}%')
print(f'   Total cost:      ${total_costs[best_t_idx]:,.0f}  (min achievable)')

---
## Step 5b — Latency Benchmark
> *"5ms vs 5 seconds is an architectural constraint, not a math decision."*

In [ ]:
import time

# Simulate real-time single-transaction scoring
single_txn = X_test.iloc[[0]]
single_txn_sc = scaler.transform(single_txn)

N_TRIALS = 1000

# XGBoost latency
start = time.perf_counter()
for _ in range(N_TRIALS):
    _ = xgb_smote.predict_proba(single_txn)
xgb_ms = (time.perf_counter() - start) / N_TRIALS * 1000

# LR latency
start = time.perf_counter()
for _ in range(N_TRIALS):
    _ = lr.predict_proba(single_txn_sc)
lr_ms = (time.perf_counter() - start) / N_TRIALS * 1000

print('⏱️  Single-transaction scoring latency (avg over 1,000 trials):')
print(f'   Logistic Regression:  {lr_ms:.3f} ms')
print(f'   XGBoost + SMOTE:      {xgb_ms:.3f} ms')
print(f'\n   LLM API call (typical): ~500–2000 ms')
print(f'   Budget for real-time fraud scoring: <20 ms')
print(f'\n✅  XGBoost easily fits the real-time constraint.')

---
## Step 6 — Explainability: Three Levels
> *"Even a perfect model fails if no one trusts it. In a regulated industry, explainability is not optional — it's a requirement."*

### Global: What does the model rely on overall?
### Local: Why did THIS transaction get flagged?
### Operational: Can a human act on the explanation?

In [ ]:
# ── GLOBAL SHAP — Feature Importance ──────────────────────────────────────────
print('Computing SHAP values (this takes ~15s on 50k rows — sampling 2k for display)...')

# Sample for speed
SHAP_N = 2000
X_shap  = X_test.sample(SHAP_N, random_state=SEED)
y_shap  = y_test.loc[X_shap.index]

explainer  = shap.TreeExplainer(xgb_smote)
shap_vals  = explainer.shap_values(X_shap)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Step 6 — Global Explainability: SHAP Feature Importance',
             fontsize=12, fontweight='bold', color='white')

# Mean |SHAP| bar chart
ax = axes[0]
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
order = np.argsort(mean_abs_shap)
feat_names = FEATURES
colors_bar = [C['mint'] if v == mean_abs_shap.max() else C['teal']
              for v in mean_abs_shap[order]]
ax.barh([feat_names[i] for i in order], mean_abs_shap[order],
        color=colors_bar, edgecolor='none', height=0.6)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Global Feature Importance', color='white')

# SHAP beeswarm (via matplotlib to stay in dark theme)
ax = axes[1]
top_k = 6
top_idx = np.argsort(mean_abs_shap)[::-1][:top_k]
for row_i, feat_i in enumerate(top_idx[::-1]):
    sv = shap_vals[:, feat_i]
    fv = X_shap.iloc[:, feat_i].values
    # Normalize feature value to [0,1] for color
    fv_norm = (fv - fv.min()) / (fv.max() - fv.min() + 1e-9)
    # Jitter y
    jitter = rng.uniform(-0.25, 0.25, len(sv))
    sc = ax.scatter(sv, row_i + jitter, c=fv_norm, cmap='RdYlGn_r',
                    alpha=0.25, s=6, linewidths=0)

ax.set_yticks(range(top_k))
ax.set_yticklabels([feat_names[i] for i in top_idx[::-1]])
ax.axvline(0, color=C['muted'], linewidth=0.8)
ax.set_xlabel('SHAP value (impact on fraud score)')
ax.set_title('SHAP Beeswarm — Top 6 Features', color='white')
plt.colorbar(sc, ax=ax, label='Feature value (low → high)')

plt.tight_layout()
plt.show()

In [ ]:
# ── LOCAL SHAP — Waterfall for a Single Flagged Transaction ───────────────────
# Find a high-scoring actual fraud case
fraud_idx    = y_shap[y_shap == 1].index
fraud_scores = pd.Series(xgb_smote.predict_proba(X_shap.loc[fraud_idx])[:, 1],
                         index=fraud_idx)
worst_txn_idx = fraud_scores.idxmax()
worst_txn     = X_shap.loc[worst_txn_idx]
worst_score   = fraud_scores.max()
worst_shap    = shap_vals[X_shap.index.get_loc(worst_txn_idx)]
base_val      = explainer.expected_value

# Build waterfall manually (dark-theme friendly)
feat_shap = sorted(zip(FEATURES, worst_shap), key=lambda x: abs(x[1]), reverse=True)

fig, ax = plt.subplots(figsize=(11, 5.5))
fig.patch.set_facecolor('#141D30')
ax.set_facecolor('#1E2D4E')

names  = [f[0] for f in feat_shap]
values = [f[1] for f in feat_shap]
bar_colors = [C['red'] if v > 0 else C['mint'] for v in values]

# Bar chart of SHAP contributions
bars = ax.barh(names[::-1], [abs(v) for v in values[::-1]],
               color=[C['red'] if v > 0 else C['mint'] for v in values[::-1]],
               height=0.55, edgecolor='none')

# Annotate with value + feature value
for i, (bar, (feat, sv)) in enumerate(zip(bars, zip(names[::-1], values[::-1]))):
    feat_val = worst_txn[feat]
    sign = '+' if sv > 0 else '−'
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{sign}{abs(sv):.3f}   ({feat_val:.2f})',
            va='center', ha='left', fontsize=8.5, color='white')

ax.set_xlabel('|SHAP contribution to fraud score|')
txn_row = df.loc[worst_txn_idx]
ax.set_title(
    f'Local Explainability — TXN-{txn_row["txn_id"]}  |  '
    f'Fraud Score: {worst_score:.3f} → FLAGGED\n'
    f'Amount: ${txn_row["txn_amount"]:,.2f}  |  '
    f'Distance: {txn_row["geo_distance_mi"]:,.0f} mi  |  '
    f'Velocity: {txn_row["txn_velocity_1h"]} txns/hr  |  '
    f'Hour: {int(txn_row["hour_of_day"]):02d}:xx',
    color='white', fontsize=10, pad=10
)

red_patch   = mpatches.Patch(color=C['red'],  label='Increases fraud score')
green_patch = mpatches.Patch(color=C['mint'], label='Decreases fraud score')
ax.legend(handles=[red_patch, green_patch], fontsize=8.5,
          facecolor='#2A3D5A', edgecolor='none', labelcolor='white')

plt.tight_layout()
plt.show()

# Operational explanation (what a banker would say)
top_pos = [(n, v) for n, v in zip(names, values) if v > 0][:3]
print(f'\n🏦 Operational Explanation (what the banker tells the customer):')
print(f'   "We flagged transaction TXN-{txn_row["txn_id"]} because:')
for feat, sv in top_pos:
    human = {
        'txn_velocity_1h':  f'{int(txn_row["txn_velocity_1h"])} transactions in the last hour on your card',
        'geo_distance_mi':  f'this transaction is {txn_row["geo_distance_mi"]:,.0f} miles from your usual area',
        'merchant_risk':    f'this merchant category has elevated fraud risk',
        'amount_vs_avg':    f'this amount is {txn_row["amount_vs_avg"]:.1f}x your typical spend',
        'txn_amount':       f'the transaction amount (${txn_row["txn_amount"]:,.2f}) is unusually large',
        'intl_flag':        'this appears to be an international transaction',
        'hour_of_day':      f'it occurred at an unusual time ({int(txn_row["hour_of_day"]):02d}:xx)',
        'account_age_days': f'the account is {txn_row["account_age_days"]} days old',
    }.get(feat, feat)
    print(f'   → {human}')
print(f'   If this was you, please verify at [link]."')

---
## Bonus — Sampling Effect: What Happens Without SMOTE?
> *"Bad sample → bad model. Show the class."*

In [ ]:
# ── SHOW THE COST OF IGNORING CLASS IMBALANCE ─────────────────────────────────
xgb_naive_params = {**xgb_params, 'scale_pos_weight': 1}  # no class weighting
xgb_naive = xgb.XGBClassifier(**xgb_naive_params)
xgb_naive.fit(X_train, y_train)  # no SMOTE, no weight
naive_proba = xgb_naive.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('Sampling Effect: Balanced vs. Naive Training',
             fontsize=12, fontweight='bold', color='white')

model_pair = [
    ('XGBoost (naive, no balancing)', naive_proba,     C['muted']),
    ('XGBoost + SMOTE (balanced)',    xgb_smote_proba, C['mint']),
]

# PR Curves
ax = axes[0]
for name, proba, color in model_pair:
    p, r, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(r, p, color=color, linewidth=2.5, label=f'{name}\nPR-AUC = {ap:.4f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR Curve: Does Balancing Help?', color='white')
ax.legend(fontsize=8.5, facecolor='#2A3D5A', edgecolor='none', labelcolor='white')

# Score distributions
ax = axes[1]
for name, proba, color in model_pair:
    fraud_scores_here = proba[y_test == 1]
    ax.hist(fraud_scores_here, bins=40, density=True, alpha=0.55,
            color=color, label=f'{name.split("(")[0].strip()}')
ax.set_xlabel('Predicted Score for FRAUD cases')
ax.set_ylabel('Density')
ax.set_title('How Well Does Each Model Score Fraud?', color='white')
ax.legend(fontsize=9, facecolor='#2A3D5A', edgecolor='none', labelcolor='white')

plt.tight_layout()
plt.show()

naive_ap  = average_precision_score(y_test, naive_proba)
smote_ap  = average_precision_score(y_test, xgb_smote_proba)
print(f'\n📊 Impact of Sampling:')
print(f'   XGBoost (naive):       PR-AUC = {naive_ap:.4f}')
print(f'   XGBoost + SMOTE:       PR-AUC = {smote_ap:.4f}')
print(f'   Improvement from sampling: +{(smote_ap - naive_ap)*100:.2f} pp')
print(f'\n💡 The algorithm is the same. The sampling is different.')
print(f'   This is why Step 3 (Sampling) matters as much as Step 4 (Model).')

---
## Summary — The Full Recipe

| Step | What We Did | Why It Matters |
|------|------------|----------------|
| **Problem** | Binary fraud classification | Chose PR-AUC, not accuracy |
| **Data** | Faker + make_classification | Realistic imbalance, label noise |
| **Sampling** | SMOTE at 15% + temporal split | No leakage, balanced learning |
| **Model** | XGBoost (scale_pos_weight) | Fast, interpretable, tabular SOTA |
| **Evaluation** | PR-AUC + KS + threshold + cost | Not one number — a system |
| **Explainability** | SHAP global + waterfall | Trusted by analysts and regulators |

**Key lines:**
- *"Getting to 99% accuracy is easy. Getting the RIGHT 99% — at the RIGHT latency — WITH explanations — is the whole game."*
- *"The algorithm is the same. The sampling is different."*
- *"The AI said so is not an answer in a regulated industry."*